# Session 3: Open-Ended Exploration — Type 1 vs Type 2 Diabetes — INSTRUCTOR VERSION

## Goals

- Ask **at least 2 clinical questions** of your own
- **At least 1** must compare Type 1 and Type 2 diabetes
- Find **at least 1 surprise or failure** in the agent's behavior
- Visualize your findings
- Submit a structured report

## About the Patient Population

This FHIR server contains **1,027 synthetic patients** generated with clinically coupled phenotypes. The patients are distributed across **6 clinical groups**:

| # | Phenotype | Description |
|---|-----------|-------------|
| 1 | **Metabolic syndrome** | Elevated BMI, blood pressure, triglycerides, and fasting glucose — but no diabetes diagnosis yet. |
| 2 | **Early Type 2 diabetes** | Recently diagnosed Type 2 diabetes with mildly elevated HbA1c. Kidney function is normal. |
| 3 | **Type 2 diabetes with chronic kidney disease stage G2** | Established Type 2 diabetes with mildly reduced kidney function (eGFR 60–89). |
| 4 | **Advanced Type 2 diabetes with chronic kidney disease stage G3b** | Long-standing Type 2 diabetes with moderately-to-severely reduced kidney function (eGFR 30–44). Often on insulin and multiple medications. |
| 5 | **Type 1 diabetes with early nephropathy** | Type 1 diabetes (the body's immune system destroys insulin-producing cells) with early signs of kidney damage. Very low C-peptide. |
| 6 | **Type 1 diabetes with poor control and chronic kidney disease stage G3a** | Type 1 diabetes with poor glycemic control (high HbA1c) and moderately reduced kidney function (eGFR 45–59). |

These phenotypes are **clinically coupled** — patients with worse diabetes control tend to have worse kidney function, mirroring real-world patterns.

## Clinical Code Reference

### Diagnosis Codes (SNOMED CT)

| Code | Condition | Notes |
|------|-----------|-------|
| 44054006 | Type 2 diabetes mellitus | The body becomes resistant to insulin |
| 46635009 | Type 1 diabetes mellitus | The immune system destroys insulin-producing cells |
| 709044004 | Chronic kidney disease | Gradual loss of kidney function over time |

### Observation Codes (LOINC)

| Code | Test | What It Measures |
|------|------|-----------------|
| 4548-4 | Hemoglobin A1c (HbA1c) | Average blood sugar over 2–3 months |
| 2160-0 | Creatinine | Waste product filtered by kidneys |
| 33914-3 | Estimated glomerular filtration rate (eGFR) | How well kidneys filter blood |
| 14959-1 | Urine albumin/creatinine ratio (UACR) | Protein leakage indicating kidney damage |
| 1986-9 | C-peptide | Marker of insulin production by the pancreas |
| 85354-9 | Blood pressure panel | Systolic and diastolic blood pressure |
| 39156-5 | Body mass index (BMI) | Weight relative to height |
| 1558-6 | Fasting glucose | Blood sugar after overnight fasting |
| 2339-0 | Blood glucose | Random blood sugar measurement |
| 3094-0 | Blood urea nitrogen (BUN) | Another kidney function marker |
| 13457-7 | LDL cholesterol | "Bad" cholesterol |
| 2085-9 | HDL cholesterol | "Good" cholesterol |
| 2571-8 | Triglycerides | Blood fat linked to heart disease risk |

### Interpretation Thresholds

| Measure | Range | Interpretation |
|---------|-------|----------------|
| HbA1c | < 5.7% | Normal |
| | 5.7% – 6.4% | Prediabetes |
| | ≥ 6.5% | Diabetes |
| | **> 7.5%** | **Poor glycemic control** |
| eGFR | > 90 | Normal kidney function |
| | 60–89 | Mildly decreased |
| | 45–59 | Moderately decreased |
| | 30–44 | Moderately-to-severely decreased |
| | < 30 | Severely decreased |
| UACR | < 30 mg/g | Normal |
| | 30–300 mg/g | Moderately increased albuminuria |
| | > 300 mg/g | Severely increased albuminuria |

## Type 1 vs Type 2 Diabetes — A Quick Guide

Understanding the difference between Type 1 and Type 2 diabetes is essential for interpreting this patient population.

**Type 1 diabetes** — the body's immune system destroys the insulin-producing beta cells in the pancreas. Patients:
- Have **very low C-peptide** (a marker of insulin production)
- Are **insulin-dependent** from diagnosis
- Tend to be diagnosed younger
- Are often leaner (lower BMI)

**Type 2 diabetes** — the body becomes resistant to insulin. Patients:
- Have **preserved or elevated C-peptide** (the pancreas still produces insulin)
- Often start with oral medications (metformin, SGLT2 inhibitors, GLP-1 receptor agonists) before needing insulin
- Are often overweight (higher BMI)
- Tend to be diagnosed later in life

**Both types** can develop chronic kidney disease, but through somewhat different pathways and timelines.

**Key differentiating lab:** C-peptide (LOINC 1986-9) is the clearest discriminator between the two types. Low C-peptide suggests Type 1; preserved or high C-peptide suggests Type 2.

In [ ]:
!pip install anthropic requests pandas matplotlib

In [ ]:
import os, json, requests, urllib3
import pandas as pd
import matplotlib.pyplot as plt
from anthropic import Anthropic

# Suppress SSL warnings (self-signed cert on teaching server)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---- Anthropic Client ----
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except (ImportError, Exception):
    api_key = None

api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError(
        "Set ANTHROPIC_API_KEY in Colab Secrets (key icon in left sidebar) "
        "or as an environment variable."
    )

client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-4-20250514"

# ---- FHIR Server ----
FHIR_BASE = "https://lfh-fhir.eastus2.cloudapp.azure.com:9443/fhir-server/api/v4"
FHIR_SESSION = requests.Session()
FHIR_SESSION.auth = ("fhiruser", "BmI512@ccess")
FHIR_SESSION.verify = False

# Verify connections
resp = FHIR_SESSION.get(f"{FHIR_BASE}/metadata", params={"_format": "json"}, timeout=10)
if resp.status_code == 200:
    fhir_version = resp.json().get("fhirVersion", "unknown")
    print(f"\u2705 FHIR server connected (version {fhir_version})")
    count_resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient",
        params={"_summary": "count", "_format": "json"},
        timeout=10
    )
    if count_resp.status_code == 200:
        total = count_resp.json().get("total", "unknown")
        print(f"   {total} patients available")
else:
    print(f"\u274c FHIR server error: HTTP {resp.status_code}")

print(f"\u2705 Anthropic client ready (model: {MODEL})")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FHIR Tool Functions
# ══════════════════════════════════════════════════════════════

def search_conditions(code: str, max_results: int = 50) -> dict:
    """Search for conditions by SNOMED CT code."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Condition",
        params={"code": code, "_count": max_results, "_format": "json"},
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id"),
            "code": coding.get("code", ""),
            "display": coding.get("display", ""),
            "patient_reference": r.get("subject", {}).get("reference", ""),
            "clinical_status": r.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", ""),
        })
    return {"total": bundle.get("total"), "results": results}


def get_patient(patient_id: str) -> dict:
    """Get a single patient's demographics."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient/{patient_id}",
        params={"_format": "json"},
        timeout=30,
    )
    p = resp.json()
    name = p.get("name", [{}])[0]
    return {
        "id": p.get("id"),
        "name": f"{' '.join(name.get('given', []))} {name.get('family', '')}".strip(),
        "gender": p.get("gender", ""),
        "birthDate": p.get("birthDate", ""),
    }


def search_observations(patient_id: str, loinc_code: str, max_results: int = 5) -> dict:
    """Search for observations by patient and LOINC code."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Observation",
        params={
            "subject": f"Patient/{patient_id}",
            "code": loinc_code,
            "_count": max_results,
            "_sort": "-date",
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        value_qty = r.get("valueQuantity", {})
        result_entry = {
            "observation_id": r.get("id"),
            "code": r.get("code", {}).get("coding", [{}])[0].get("code", ""),
            "display": r.get("code", {}).get("coding", [{}])[0].get("display", ""),
            "value": value_qty.get("value"),
            "unit": value_qty.get("unit", ""),
            "date": r.get("effectiveDateTime", ""),
        }
        # Handle component-based observations (e.g., blood pressure)
        if not value_qty.get("value") and r.get("component"):
            components = []
            for comp in r["component"]:
                comp_code = comp.get("code", {}).get("coding", [{}])[0]
                comp_val = comp.get("valueQuantity", {})
                components.append({
                    "component": comp_code.get("display", ""),
                    "value": comp_val.get("value"),
                    "unit": comp_val.get("unit", ""),
                })
            result_entry["components"] = components
        results.append(result_entry)
    return {"total": bundle.get("total"), "results": results}


def search_medications(patient_id: str, max_results: int = 10) -> dict:
    """Search for medication requests for a patient."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/MedicationRequest",
        params={
            "subject": f"Patient/{patient_id}",
            "_count": max_results,
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        med_concept = r.get("medicationCodeableConcept", {})
        coding = med_concept.get("coding", [{}])[0] if med_concept.get("coding") else {}
        med_name = coding.get("display") or med_concept.get("text", "unknown")
        results.append({
            "medication": med_name,
            "code": coding.get("code", ""),
            "status": r.get("status", ""),
            "date": r.get("authoredOn", ""),
        })
    return {"total": bundle.get("total"), "results": results}


def search_all_conditions(patient_id: str, max_results: int = 20) -> dict:
    """Get all conditions (full problem list) for a specific patient."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Condition",
        params={
            "subject": f"Patient/{patient_id}",
            "_count": max_results,
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id"),
            "code": coding.get("code", ""),
            "display": coding.get("display", ""),
            "clinical_status": r.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", ""),
        })
    return {"total": bundle.get("total"), "results": results}


# ---- Smoke tests ----
print("Testing tool functions...")
_test = search_conditions("44054006", max_results=3)
print(f"  search_conditions('44054006'): {_test['total']} conditions found")
if _test["results"]:
    _pid = _test["results"][0]["patient_reference"].split("/")[-1]
    _p = get_patient(_pid)
    print(f"  get_patient('{_pid}'): {_p['name']}")
    _obs = search_observations(_pid, "4548-4", max_results=1)
    print(f"  search_observations(HbA1c): {_obs['total']} observations")
    _meds = search_medications(_pid)
    print(f"  search_medications: {_meds['total']} medications")
    _conds = search_all_conditions(_pid)
    print(f"  search_all_conditions: {_conds['total']} conditions")
print("\u2705 All tools working")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Additional Session 3 Tool Functions
# ══════════════════════════════════════════════════════════════

def search_encounters(patient_id: str, max_results: int = 10) -> dict:
    """Search for clinical encounters (visits) for a patient."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Encounter",
        params={
            "subject": f"Patient/{patient_id}",
            "_count": max_results,
            "_sort": "-date",
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        enc_type = r.get("type", [{}])[0].get("text", "") if r.get("type") else ""
        results.append({
            "encounter_id": r.get("id"),
            "status": r.get("status", ""),
            "class": r.get("class", {}).get("code", ""),
            "type": enc_type,
            "period_start": r.get("period", {}).get("start", ""),
            "period_end": r.get("period", {}).get("end", ""),
        })
    return {"total": bundle.get("total"), "results": results}


def search_patients(max_results: int = 50) -> dict:
    """Search for patients — batch demographics retrieval."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient",
        params={"_count": max_results, "_format": "json"},
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        p = entry["resource"]
        name = p.get("name", [{}])[0]
        results.append({
            "id": p.get("id"),
            "name": f"{' '.join(name.get('given', []))} {name.get('family', '')}".strip(),
            "gender": p.get("gender", ""),
            "birthDate": p.get("birthDate", ""),
        })
    return {"total": bundle.get("total"), "results": results}

In [ ]:
# ---- Test Session 3 tools ----
print("Testing additional tools...")
if _test["results"]:
    _enc = search_encounters(_pid)
    print(f"  search_encounters: {_enc['total']} encounters")
_pts = search_patients(max_results=5)
print(f"  search_patients: {_pts['total']} total patients")
print("\u2705 All 7 tools working")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Tool Schemas — what the LLM sees
# ══════════════════════════════════════════════════════════════

tools = [
    {
        "name": "search_conditions",
        "description": (
            "Search for patient conditions by SNOMED CT diagnosis code. "
            "Returns matching conditions with patient references. "
            "Common codes: 44054006 (Type 2 diabetes mellitus), "
            "46635009 (Type 1 diabetes mellitus), "
            "709044004 (chronic kidney disease)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "SNOMED CT code (e.g., '44054006' for Type 2 diabetes)",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 50)",
                },
            },
            "required": ["code"],
        },
    },
    {
        "name": "get_patient",
        "description": (
            "Retrieve demographics for a single patient by ID. "
            "Returns name, gender, and birth date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID (from a condition's patient_reference, e.g., 'abc123')",
                },
            },
            "required": ["patient_id"],
        },
    },
    {
        "name": "search_observations",
        "description": (
            "Search for clinical observations (lab results, vitals) for a patient "
            "by LOINC code. Returns values sorted by date (most recent first). "
            "Common codes: 4548-4 (HbA1c), 2160-0 (creatinine), 33914-3 (eGFR), "
            "1986-9 (C-peptide), 14959-1 (urine albumin/creatinine ratio), "
            "85354-9 (blood pressure), 39156-5 (BMI), 1558-6 (fasting glucose), "
            "2339-0 (blood glucose), 3094-0 (BUN), 13457-7 (LDL cholesterol), "
            "2085-9 (HDL cholesterol), 2571-8 (triglycerides)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "loinc_code": {
                    "type": "string",
                    "description": "LOINC code for the observation (e.g., '4548-4' for HbA1c)",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 5)",
                },
            },
            "required": ["patient_id", "loinc_code"],
        },
    },
    {
        "name": "search_medications",
        "description": (
            "Search for medication prescriptions (MedicationRequest resources) "
            "for a patient. Returns medication name, status, and date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 10)",
                },
            },
            "required": ["patient_id"],
        },
    },
    {
        "name": "search_all_conditions",
        "description": (
            "Retrieve the complete problem list (all diagnoses) for a specific "
            "patient. Unlike search_conditions which finds patients by one "
            "diagnosis code, this returns ALL conditions for one patient."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 20)",
                },
            },
            "required": ["patient_id"],
        },
    },
]

available_functions = {
    "search_conditions": search_conditions,
    "get_patient": get_patient,
    "search_observations": search_observations,
    "search_medications": search_medications,
    "search_all_conditions": search_all_conditions,
}

# ── Add Session 3 tools ──────────────────────────────────────

tools.extend([
    {
        "name": "search_encounters",
        "description": (
            "Search for clinical encounters (visits) for a patient. "
            "Returns encounter type, status, class (ambulatory, emergency, "
            "inpatient), and dates."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 10)",
                },
            },
            "required": ["patient_id"],
        },
    },
    {
        "name": "search_patients",
        "description": (
            "Search for patients on the server. Returns demographics (name, "
            "gender, birth date) for multiple patients at once. Useful for "
            "getting an overview of the patient population."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 50)",
                },
            },
            "required": [],
        },
    },
])

available_functions["search_encounters"] = search_encounters
available_functions["search_patients"] = search_patients

print(f"\u2705 {len(tools)} tools registered: {[t['name'] for t in tools]}")

In [ ]:

SYSTEM_PROMPT = \"\"\"You are a clinical data assistant with access to a FHIR \
server containing synthetic patient records.

PATIENT POPULATION: The server has approximately 1,027 patients across 6 \
clinical phenotypes:
1. Metabolic syndrome
2. Early Type 2 diabetes
3. Type 2 diabetes with chronic kidney disease stage G2
4. Advanced Type 2 diabetes with chronic kidney disease stage G3b
5. Type 1 diabetes with early nephropathy
6. Type 1 diabetes with poor control and chronic kidney disease stage G3a

DIAGNOSIS CODES (SNOMED CT):
- 44054006: Type 2 diabetes mellitus
- 46635009: Type 1 diabetes mellitus
- 709044004: Chronic kidney disease

OBSERVATION CODES (LOINC):
- 4548-4: Hemoglobin A1c (HbA1c) — glycemic control marker
- 2160-0: Creatinine — kidney function marker
- 33914-3: Estimated glomerular filtration rate (eGFR) — kidney function
- 14959-1: Urine albumin/creatinine ratio (UACR) — kidney damage marker
- 1986-9: C-peptide — insulin production marker (low in Type 1, normal/high in Type 2)
- 85354-9: Blood pressure panel
- 39156-5: Body mass index (BMI)
- 1558-6: Fasting glucose
- 2339-0: Blood glucose
- 3094-0: Blood urea nitrogen (BUN)
- 13457-7: LDL cholesterol
- 2085-9: HDL cholesterol
- 2571-8: Triglycerides

INTERPRETATION THRESHOLDS:
- HbA1c > 7.5%: poor glycemic control
- eGFR < 60 mL/min/1.73m²: impaired kidney function
- eGFR < 30: severely impaired kidney function
- UACR > 30 mg/g: moderately increased albuminuria
- UACR > 300 mg/g: severely increased albuminuria

AVAILABLE TOOLS (7):
You have tools to search conditions by diagnosis code, get individual patient \
demographics, search observations by LOINC code, search medications, get a \
patient's full problem list, search encounters (visits), and batch-search \
patients. Use the right tool for the right task.

STRATEGY — think step by step:
1. Identify the condition(s) relevant to the question using search_conditions
2. Extract patient references from the condition results
3. Get patient demographics with get_patient (or search_patients for batch)
4. Look up relevant observations using the appropriate LOINC codes
5. Check medications and/or encounters if relevant
6. Review the full problem list (search_all_conditions) if comorbidities matter
7. Synthesize findings into a clear clinical summary

RULES:
- NEVER invent or hallucinate data. Only report values actually returned by tools.
- If a tool returns no results for a patient, state that explicitly.
- Always identify patients by name when available.
- Show actual lab values, not just categories.
- When comparing groups, provide counts and specific values.
- Be thorough: check all relevant patients, not just the first few.
\"\"\"


In [ ]:

def run_agent(question, system_prompt=SYSTEM_PROMPT, tools=tools,
              available_functions=available_functions, max_steps=25):
    \"\"\"Run the tool-use agent loop.\"\"\"
    print(f"\U0001f916 AGENT QUESTION: {question}\n")
    print("=" * 70)

    messages = [{"role": "user", "content": question}]
    tool_calls_log = []
    step = 0

    while step < max_steps:
        step += 1
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )

        # Check for tool use
        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

        if not tool_use_blocks:
            # Final text response
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            print(f"\n{'=' * 70}")
            print(f"\u2705 FINAL ANSWER (after {step} steps):\n")
            print(final_text)
            return final_text, tool_calls_log, messages

        # Serialize assistant content for message history
        assistant_content = []
        for block in response.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                assistant_content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input,
                })
        messages.append({"role": "assistant", "content": assistant_content})

        # Execute each tool call
        tool_results = []
        for block in tool_use_blocks:
            fn_name = block.name
            fn_args = block.input

            args_str = ", ".join(f"{k}={v!r}" for k, v in fn_args.items())
            print(f"\U0001f527 Step {step}: {fn_name}({args_str})")

            tool_calls_log.append({
                "step": step,
                "function": fn_name,
                "arguments": fn_args,
            })

            try:
                result = available_functions[fn_name](**fn_args)
            except Exception as e:
                result = {"error": str(e)}

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result, default=str),
            })

        messages.append({"role": "user", "content": tool_results})

    print(f"\n\u26a0\ufe0f Reached max steps ({max_steps})")
    return "Agent reached step limit.", tool_calls_log, messages


## Question Ideas

Choose from these or create your own. You need **at least 2**, with **at least 1 comparing Type 1 and Type 2 diabetes**.

### Straightforward
- "Find patients with Type 1 diabetes (SNOMED 46635009) and retrieve their HbA1c and C-peptide values"
- "Find patients with chronic kidney disease (SNOMED 709044004) and check their eGFR and creatinine"

### Type 1 vs Type 2 Comparisons (pick at least one)
- "Compare HbA1c levels between Type 1 and Type 2 diabetes patients. Which group has worse glycemic control?"
- "Compare C-peptide levels between Type 1 and Type 2 diabetes patients. How do they differ and why?"
- "Compare medication patterns between Type 1 and Type 2 diabetes patients"

### Complex / Multi-Step
- "Find patients with both diabetes and chronic kidney disease. Is there a relationship between their HbA1c and eGFR?"
- "Which patients have the worst kidney function? What are their other diagnoses and medications?"

### Edge Cases (likely to produce agent failures)
- "Are there patients who might be misclassified — diagnosed with Type 2 diabetes but with C-peptide levels more consistent with Type 1?"
- "Find the healthiest patients on this server" (vague — interesting failure mode)

After running your agent questions, you can visualize the data:
```python
# 1. Collect data using the tool functions directly
# 2. Build a pandas DataFrame
# 3. Plot with matplotlib
# Example: plt.scatter(df['c_peptide'], df['hba1c'])
```

## Question 1

Enter your first question in the cell below. Describe what you expect the agent to do before running it.

In [ ]:
user_question_1 = ""
# ^^^ Fill in your question above ^^^

answer_1, tool_calls_1, messages_1 = run_agent(user_question_1)

In [ ]:
# Question 1 trace
tool_calls_log = tool_calls_1  # for the trace display

# ── Tool Call Trace ──────────────────────────────────────────
print("\U0001f4cb TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)
for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\nTotal tool calls: {len(tool_calls_log)}")
tools_used = set(tc["function"] for tc in tool_calls_log)
print(f"Unique tools used: {sorted(tools_used)}")


## Question 2 — Type 1 vs Type 2 Comparison

If you haven't already, make this one a **comparison between Type 1 and Type 2 diabetes**. The agent has access to SNOMED codes for both (44054006 for Type 2, 46635009 for Type 1) and C-peptide (LOINC 1986-9) is the key differentiating lab.

In [ ]:
user_question_2 = ""
# ^^^ Fill in your question above ^^^

answer_2, tool_calls_2, messages_2 = run_agent(user_question_2)

In [ ]:
# Question 2 trace
tool_calls_log = tool_calls_2  # for the trace display

# ── Tool Call Trace ──────────────────────────────────────────
print("\U0001f4cb TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)
for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\nTotal tool calls: {len(tool_calls_log)}")
tools_used = set(tc["function"] for tc in tool_calls_log)
print(f"Unique tools used: {sorted(tools_used)}")


## Visualize Your Findings

The cell below creates a pre-built comparison plot of Type 1 vs Type 2 diabetes patients using C-peptide and HbA1c values. Run it to see the biochemical difference between the two types.

You can also create your own plots using the tool functions directly — for example:

```python
# Get data for your specific question
results = search_observations(patient_id, "4548-4")
# Build a DataFrame and plot
df = pd.DataFrame(rows)
plt.scatter(df['x_column'], df['y_column'])
plt.show()
```

In [ ]:

# ══════════════════════════════════════════════════════════════
# COMPARISON PLOT: Type 1 vs Type 2 Diabetes
# ══════════════════════════════════════════════════════════════
# This cell queries both diabetes groups and plots C-peptide vs HbA1c
# to visualize the key biochemical difference between them.

# Fetch both groups
t2d = search_conditions("44054006", max_results=50)
t2d_ids = list(set(r["patient_reference"].split("/")[-1] for r in t2d["results"]))

t1d = search_conditions("46635009", max_results=50)
t1d_ids = list(set(r["patient_reference"].split("/")[-1] for r in t1d["results"]))

print(f"Type 2 diabetes patients: {len(t2d_ids)}")
print(f"Type 1 diabetes patients: {len(t1d_ids)}")

# Collect C-peptide and HbA1c for both groups
rows = []
for diabetes_type, patient_ids in [("Type 2", t2d_ids), ("Type 1", t1d_ids)]:
    for pid in patient_ids:
        hba1c = search_observations(pid, "4548-4", max_results=1)
        cpeptide = search_observations(pid, "1986-9", max_results=1)
        rows.append({
            "patient_id": pid,
            "diabetes_type": diabetes_type,
            "hba1c": hba1c["results"][0]["value"] if hba1c["results"] else None,
            "c_peptide": cpeptide["results"][0]["value"] if cpeptide["results"] else None,
        })

df_compare = pd.DataFrame(rows)
df_compare["hba1c"] = pd.to_numeric(df_compare["hba1c"], errors="coerce")
df_compare["c_peptide"] = pd.to_numeric(df_compare["c_peptide"], errors="coerce")

plot_df = df_compare.dropna(subset=["hba1c", "c_peptide"])
if len(plot_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))

    for dtype, color, marker in [("Type 1", "tab:blue", "s"), ("Type 2", "tab:orange", "o")]:
        subset = plot_df[plot_df["diabetes_type"] == dtype]
        ax.scatter(subset["c_peptide"], subset["hba1c"], c=color, marker=marker,
                   label=f"{dtype} diabetes", alpha=0.7, edgecolors="black",
                   linewidth=0.5, s=80)

    ax.axhline(y=7.5, color="gray", linestyle=":", alpha=0.7, label="HbA1c = 7.5%")
    ax.set_xlabel("C-Peptide (ng/mL)", fontsize=12)
    ax.set_ylabel("HbA1c (%)", fontsize=12)
    ax.set_title("C-Peptide vs HbA1c: Type 1 vs Type 2 Diabetes", fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    for dtype in ["Type 1", "Type 2"]:
        sub = plot_df[plot_df["diabetes_type"] == dtype]
        print(f"\n{dtype} diabetes ({len(sub)} patients):")
        print(f"  C-peptide: mean={sub['c_peptide'].mean():.2f}, range={sub['c_peptide'].min():.2f}-{sub['c_peptide'].max():.2f}")
        print(f"  HbA1c: mean={sub['hba1c'].mean():.1f}%, range={sub['hba1c'].min():.1f}-{sub['hba1c'].max():.1f}%")
else:
    print("Not enough data for comparison plot. Check that both patient groups have C-peptide and HbA1c data.")


## Failure and Surprise Analysis

**What surprised you about the agent's behavior?**

Common surprises: the agent may not search for both diabetes types in a comparison question (only doing one, then answering). It may hallucinate a LOINC code for a lab it doesn't have. With 7 tools, it has more choices and may pick suboptimal ones.

**Did the agent make any errors?**

Watch for: inventing data not returned by tools, incomplete searches (checking only a few patients), wrong SNOMED codes, premature stopping before checking all patients.

**How would you improve the agent?**

Options: refine the system prompt to be more explicit about comparison strategies, add tool descriptions that clarify when to use `search_all_conditions` vs `search_conditions`, increase `max_steps` for complex questions.

In [ ]:

# ══════════════════════════════════════════════════════════════
# DELIVERABLE — Session 3 Report
# ══════════════════════════════════════════════════════════════
from datetime import datetime

report = {
    "student_id": input("Enter your student ID: "),
    "timestamp": datetime.now().isoformat(),
    "session": 3,
    "runs": [],
}

# Add Question 1 if available
try:
    report["runs"].append({
        "question": user_question_1,
        "tool_calls": tool_calls_1,
        "num_tool_calls": len(tool_calls_1),
        "tools_used": list(set(tc["function"] for tc in tool_calls_1)),
        "final_answer": answer_1[:500] if isinstance(answer_1, str) else "",
    })
except NameError:
    print("\u26a0\ufe0f Question 1 not found \u2014 did you run it?")

# Add Question 2 if available
try:
    report["runs"].append({
        "question": user_question_2,
        "tool_calls": tool_calls_2,
        "num_tool_calls": len(tool_calls_2),
        "tools_used": list(set(tc["function"] for tc in tool_calls_2)),
        "final_answer": answer_2[:500] if isinstance(answer_2, str) else "",
    })
except NameError:
    print("\u26a0\ufe0f Question 2 not found \u2014 did you run it?")

# Save
filename = f"session3_report_{report['student_id']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(filename, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"\n\u2705 Report saved: {filename}")
print(f"   Questions answered: {len(report['runs'])}")
print(f"   Total tool calls: {sum(r['num_tool_calls'] for r in report['runs'])}")
print(f"\nDownload this file and submit it.")


## Final Reflection

Across all three sessions, you have:

1. **Queried FHIR data manually** — understanding the API, references, and clinical codes
2. **Watched an AI agent** perform the same workflow autonomously using tool schemas
3. **Tested the agent** with your own questions and found its limitations

The tools and patterns from these sessions — FHIR REST APIs, structured clinical terminologies (SNOMED, LOINC), LLM tool use, and agent evaluation — are the building blocks of modern clinical AI systems.

**Key takeaway:** AI agents are powerful but not infallible. Domain expertise (understanding the clinical question, the data, and the expected answer) is what makes evaluation possible. The agent does the data fetching; the human ensures the answer makes clinical sense.